# 04 · Ajuste de hiperparámetros de los 3 mejores candidatos

**Método: Optuna (muestreador TPE, semilla 42), 30 configuraciones por modelo.** Justificación:
1. Nuestra CV **recalcula el target en cada pliegue** (umbral P90 con el entrenamiento del pliegue). `RandomizedSearchCV`/`GridSearchCV` asumen un `y` fijo, así que no pueden reproducir este esquema sin fuga; con Optuna la función objetivo es exactamente `evaluar_cv`, la misma CV de los notebooks 02 y 03.
2. Cada configuración cuesta 5 entrenamientos; con un presupuesto fijo de 30, TPE concentra las pruebas en las regiones prometedoras en vez de muestrear a ciegas (búsqueda aleatoria) o explotar combinatoriamente (búsqueda exhaustiva).
3. Las distribuciones log-uniformes son nativas (`log=True`).

**Registro en MLflow.** Una corrida **padre** por modelo (`fase=tuning`) y una corrida **hija** anidada por configuración (parámetros, datos, métricas de CV por pliegue, tiempos, tags). El pipeline entrenado se guarda en la corrida padre para la mejor configuración (reentrenada con todo el entrenamiento); no se guardan 90 modelos (un Random Forest pesa cientos de MB).

**Criterio de mejora** (enunciado): la mejora del PR-AUC medio en CV respecto al modelo por defecto debe superar la desviación entre pliegues del modelo por defecto; si no, no se considera mejora.

In [ ]:
import sys, warnings
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
sns.set_theme(style="whitegrid", context="notebook")
FIG = RAIZ / "docs" / "figuras"
FIG.mkdir(parents=True, exist_ok=True)

def guardar(fig, nombre):
    fig.savefig(FIG / f"{nombre}.png", dpi=130, bbox_inches="tight")

In [ ]:
import time
import optuna
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier

from src.datos import cargar_obt, construir_dataset, dividir_train_test
from src.pipeline import (SEMILLA, EXPERIMENTO, crear_pipeline, evaluar_cv, registrar_corrida,
                          registrar_en_corrida_activa, configurar_mlflow, contexto_git)

optuna.logging.set_verbosity(optuna.logging.WARNING)
obt, meta = cargar_obt()
train, test = dividir_train_test(construir_dataset(obt))
del test
mlflow = configurar_mlflow()
top3 = pd.read_csv(RAIZ / "docs" / "top3_para_tuning.csv")
N_CONFIGS = 30
print("Git:", contexto_git())
top3

## 1. Espacios de búsqueda

| Modelo | Hiperparámetro | Rango | Escala | Por qué |
|---|---|---|---|---|
| LogisticRegression | `C` | 1e-4 – 1e2 | log | la regularización actúa por órdenes de magnitud |
| | `penalty` | l1, l2 | — | l1 puede descartar features redundantes (los `rel_*` están correlacionados) |
| | `class_weight` | None, balanced | — | desbalance 11:1 (hallazgo 2) |
| RandomForest | `n_estimators` | 100 – 500 | lineal (paso 50) | más árboles = menos varianza, costo lineal |
| | `max_depth` | 4 – 30 | lineal | por defecto crece sin límite y memoriza (brecha del notebook 03) |
| | `min_samples_leaf` | 1 – 500 | log | hojas grandes suavizan probabilidades de una clase rara; el efecto es multiplicativo |
| | `max_features` | 0,2 – 1,0 | lineal | des-correlaciona árboles |
| | `max_samples` | 0,3 – 1,0 | lineal | bootstrap más pequeño = más diversidad y menos costo |
| | `class_weight` | None, balanced_subsample | — | desbalance |
| HistGradientBoosting | `learning_rate` | 0,01 – 0,3 | log | escala multiplicativa (ejemplo del enunciado) |
| | `max_iter` | 50 – 800 | log | interactúa con learning_rate; sin early stopping aleatorio (rompería el orden temporal) |
| | `max_leaf_nodes` | 8 – 128 | log | complejidad de cada árbol |
| | `min_samples_leaf` | 20 – 2000 | log | regularización por tamaño de hoja |
| | `l2_regularization` | 1e-4 – 10 | log | órdenes de magnitud |
| | `max_features` | 0,3 – 1,0 | lineal | submuestreo de columnas |
| | `class_weight` | None, balanced | — | desbalance |
| MLP | `hidden_layer_sizes` | (32), (64), (128), (64,32), (128,64) | — | capacidad |
| | `alpha` | 1e-6 – 1e-1 | log | penalización L2 |
| | `learning_rate_init` | 1e-4 – 1e-2 | log | escala multiplicativa |
| | `batch_size` | 128 – 1024 | — | estabilidad / costo |

Solo se ajustan los tres modelos que el notebook 03 dejó en el top 3.

In [ ]:
def espacio(modelo, trial):
    if modelo == "logistic_regression":
        return LogisticRegression(
            C=trial.suggest_float("C", 1e-4, 1e2, log=True),
            penalty=trial.suggest_categorical("penalty", ["l1", "l2"]),
            class_weight=trial.suggest_categorical("class_weight", [None, "balanced"]),
            solver="liblinear", max_iter=1000, random_state=SEMILLA)
    if modelo == "random_forest":
        return RandomForestClassifier(
            n_estimators=trial.suggest_int("n_estimators", 100, 500, step=50),
            max_depth=trial.suggest_int("max_depth", 4, 30),
            min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 500, log=True),
            max_features=trial.suggest_float("max_features", 0.2, 1.0),
            max_samples=trial.suggest_float("max_samples", 0.3, 1.0),
            class_weight=trial.suggest_categorical("class_weight", [None, "balanced_subsample"]),
            n_jobs=-1, random_state=SEMILLA)
    if modelo == "hist_gradient_boosting":
        return HistGradientBoostingClassifier(
            learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            max_iter=trial.suggest_int("max_iter", 50, 800, log=True),
            max_leaf_nodes=trial.suggest_int("max_leaf_nodes", 8, 128, log=True),
            min_samples_leaf=trial.suggest_int("min_samples_leaf", 20, 2000, log=True),
            l2_regularization=trial.suggest_float("l2_regularization", 1e-4, 10, log=True),
            max_features=trial.suggest_float("max_features", 0.3, 1.0),
            class_weight=trial.suggest_categorical("class_weight", [None, "balanced"]),
            early_stopping=False, random_state=SEMILLA)
    if modelo == "mlp":
        capas = {"32": (32,), "64": (64,), "128": (128,), "64-32": (64, 32), "128-64": (128, 64)}
        return MLPClassifier(
            hidden_layer_sizes=capas[trial.suggest_categorical("hidden_layer_sizes", list(capas))],
            alpha=trial.suggest_float("alpha", 1e-6, 1e-1, log=True),
            learning_rate_init=trial.suggest_float("learning_rate_init", 1e-4, 1e-2, log=True),
            batch_size=trial.suggest_categorical("batch_size", [128, 256, 512, 1024]),
            max_iter=200, random_state=SEMILLA)
    raise ValueError(modelo)

## 2. Búsqueda (corrida padre + 30 hijas por modelo)

In [ ]:
estudios, filas_trials, resumen = {}, [], []
for _, fila in top3.iterrows():
    modelo, familia = fila["modelo"], fila["familia"]
    defecto = mlflow.get_run(fila["run_id"]).data.metrics
    resultados_trial = {}
    t_modelo = time.perf_counter()
    with mlflow.start_run(run_name=f"tuning_{modelo}") as padre:
        def objetivo(trial):
            pipe = crear_pipeline(espacio(modelo, trial))
            r = evaluar_cv(pipe, train)
            resultados_trial[trial.number] = (pipe, r)
            registrar_corrida(f"{modelo}_config_{trial.number:02d}", pipe, train, meta,
                              fase="tuning", familia=familia, resultado=r, registrar_modelo=False,
                              nested=True, params={"optuna_trial": trial.number},
                              tags_extra={"tuning_de": modelo, "rol": "configuracion"})
            for k in ("cv_pr_auc_std", "train_pr_auc", "brecha_pr_auc", "cv_reduccion_fn_mean", "seg_fit_promedio"):
                trial.set_user_attr(k, r.resumen[k])
            return r.resumen["cv_pr_auc_mean"]

        estudio = optuna.create_study(direction="maximize", study_name=modelo,
                                      sampler=optuna.samplers.TPESampler(seed=SEMILLA))
        estudio.optimize(objetivo, n_trials=N_CONFIGS)
        estudios[modelo] = estudio

        mejor_pipe, mejor_r = resultados_trial[estudio.best_trial.number]
        mejora = mejor_r.resumen["cv_pr_auc_mean"] - defecto["cv_pr_auc_mean"]
        es_mejora = mejora > defecto["cv_pr_auc_std"]
        registrar_en_corrida_activa(
            mejor_pipe, train, meta, fase="tuning", familia=familia, resultado=mejor_r,
            params={"optuna_mejor_trial": estudio.best_trial.number, "optuna_n_configs": N_CONFIGS,
                    "optuna_sampler": "TPESampler(seed=42)", "run_id_por_defecto": fila["run_id"]},
            tags_extra={"tuning_de": modelo, "rol": "padre_mejor_configuracion"},
            metricas_extra={"defecto_cv_pr_auc_mean": defecto["cv_pr_auc_mean"],
                            "defecto_cv_pr_auc_std": defecto["cv_pr_auc_std"],
                            "mejora_vs_defecto": mejora, "mejora_supera_std": float(es_mejora),
                            "seg_tuning_total": time.perf_counter() - t_modelo})
    resumen.append({"modelo": modelo, "familia": familia, "run_id_padre": padre.info.run_id,
                    "defecto_cv": defecto["cv_pr_auc_mean"], "defecto_std": defecto["cv_pr_auc_std"],
                    "ajustado_cv": mejor_r.resumen["cv_pr_auc_mean"], "ajustado_std": mejor_r.resumen["cv_pr_auc_std"],
                    "ajustado_train": mejor_r.resumen["train_pr_auc"], "ajustado_brecha": mejor_r.resumen["brecha_pr_auc"],
                    "ajustado_reduccion_fn": mejor_r.resumen["cv_reduccion_fn_mean"],
                    "mejora": mejora, "mejora_supera_std_defecto": es_mejora,
                    "mejores_params": estudio.best_params, "minutos": (time.perf_counter() - t_modelo) / 60})
    for t in estudio.trials:
        filas_trials.append({"modelo": modelo, "config": t.number, "cv_pr_auc_mean": t.value,
                             **{f"param_{k}": v for k, v in t.params.items()}, **t.user_attrs})
    print(f"{modelo:<24} defecto {defecto['cv_pr_auc_mean']:.4f} ± {defecto['cv_pr_auc_std']:.4f} -> "
          f"ajustado {mejor_r.resumen['cv_pr_auc_mean']:.4f} ± {mejor_r.resumen['cv_pr_auc_std']:.4f} "
          f"(mejora {mejora:+.4f}; {'SUPERA' if es_mejora else 'no supera'} la desviación) "
          f"[{resumen[-1]['minutos']:.0f} min]")

## 3. Resultados del tuning

In [ ]:
res_tuning = pd.DataFrame(resumen)
trials = pd.DataFrame(filas_trials)
trials.to_csv(RAIZ / "docs" / "resultados_tuning_configs.csv", index=False)
res_tuning.to_csv(RAIZ / "docs" / "resultados_tuning.csv", index=False)
with pd.option_context("display.max_colwidth", 120):
    display(res_tuning[["modelo", "defecto_cv", "defecto_std", "ajustado_cv", "ajustado_std", "ajustado_train",
                        "ajustado_brecha", "mejora", "mejora_supera_std_defecto", "ajustado_reduccion_fn",
                        "minutos", "mejores_params"]])
print(f"Configuraciones evaluadas: {len(trials)} ({N_CONFIGS} por modelo)")

In [ ]:
fig, ax = plt.subplots(1, len(res_tuning), figsize=(6 * len(res_tuning), 4.2), sharey=True)
ax = np.atleast_1d(ax)
for a, (_, f) in zip(ax, res_tuning.iterrows()):
    t = trials[trials["modelo"] == f["modelo"]].sort_values("config")
    a.scatter(t["config"], t["cv_pr_auc_mean"], s=18, color="#4c72b0", label="configuración")
    a.plot(t["config"], t["cv_pr_auc_mean"].cummax(), color="#c44e52", label="mejor hasta ahora")
    a.axhline(f["defecto_cv"], color="k", ls="--", label="por defecto")
    a.axhspan(f["defecto_cv"] - f["defecto_std"], f["defecto_cv"] + f["defecto_std"], color="k", alpha=.08,
              label="± desv. entre pliegues (defecto)")
    a.set(title=f["modelo"], xlabel="configuración (orden de Optuna)")
ax[0].set_ylabel("PR-AUC medio en CV"); ax[0].legend(fontsize=8, loc="lower right")
fig.tight_layout(); guardar(fig, "04_tuning_historial")

In [ ]:
# Importancia de cada hiperparámetro (fANOVA de Optuna) y relación con el score
for modelo, estudio in estudios.items():
    try:
        imp = optuna.importance.get_param_importances(
            estudio, evaluator=optuna.importance.FanovaImportanceEvaluator(seed=SEMILLA))
    except Exception as e:                       # p. ej. muy pocas configuraciones
        print(f"{modelo}: fANOVA no disponible ({e}); se usa el evaluador por defecto")
        imp = optuna.importance.get_param_importances(estudio)
    params = list(imp)[:3]
    fig, ax = plt.subplots(1, len(params) + 1, figsize=(4.4 * (len(params) + 1), 3.6))
    ax[0].barh(list(imp)[::-1], list(imp.values())[::-1], color="#55a868")
    ax[0].set(title=f"{modelo}: importancia", xlabel="fANOVA")
    t = trials[trials["modelo"] == modelo]
    for a, p in zip(ax[1:], params):
        x = t[f"param_{p}"]
        if pd.api.types.is_numeric_dtype(x) and x.nunique() > 5:
            a.scatter(x, t["cv_pr_auc_mean"], s=16)
            if p in ("C", "learning_rate", "alpha", "learning_rate_init", "l2_regularization", "min_samples_leaf", "max_leaf_nodes", "max_iter"):
                a.set_xscale("log")
        else:
            t.groupby(x.astype(str))["cv_pr_auc_mean"].max().plot.bar(ax=a, color="#4c72b0")
        a.set(title=p, ylabel="PR-AUC CV")
    fig.tight_layout(); guardar(fig, f"04_tuning_params_{modelo}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(res_tuning))
ax.bar(x - 0.2, res_tuning["defecto_cv"], 0.4, yerr=res_tuning["defecto_std"], label="por defecto", color="#999999")
ax.bar(x + 0.2, res_tuning["ajustado_cv"], 0.4, yerr=res_tuning["ajustado_std"], label="ajustado", color="#4c72b0")
ax.set(xticks=x, xticklabels=res_tuning["modelo"], ylabel="PR-AUC CV (media ± desv.)",
       title="Mejora del tuning frente a la versión por defecto")
ax.legend(); fig.tight_layout(); guardar(fig, "04_tuning_mejora")